# Explore the training data

Labels, potency, agreement between repeated measurements, calculated descriptors and chemical
diversity for the p38α, COX-2 and 5-LOX datasets. Output goes to `analysis/{slug}/`.

In [ ]:
import hashlib
import json
import platform
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from rdkit import Chem, rdBase
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

WRITE_OUTPUTS = True

TARGETS = {
    'p38':  {'chembl_id': 'CHEMBL260', 'name': 'p38alpha (MAPK14)'},
    'cox2': {'chembl_id': 'CHEMBL230', 'name': 'COX-2 (PTGS2)'},
    'lox5': {'chembl_id': 'CHEMBL215', 'name': '5-LOX (ALOX5)'},
}

import os
if os.environ.get('NL_SLUGS'):
    TARGETS = {k: v for k, v in TARGETS.items() if k in os.environ['NL_SLUGS'].split(',')}

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/p38_training_raw.csv').exists()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the NeuroLead project or its src directory.')

LABELS = ['active', 'dropped', 'inactive']
COLORS = ['#198278', '#dfa43a', '#b54c68']
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})


def banner(slug):
    t = TARGETS[slug]
    display(Markdown(f'#### {t["name"]} &middot; `{t["chembl_id"]}` &middot; slug `{slug}`'))


S = {}
for slug, t in TARGETS.items():
    source = ROOT / f'data/{slug}_training_raw.csv'
    df = pd.read_csv(source)
    S[slug] = {'source': source, 'out': ROOT / f'analysis/{slug}', 'df': df}
    banner(slug)
    print(f'{source.relative_to(ROOT)}: {len(df):,} rows x {len(df.columns)} columns')
    print(f'retrieved {df.retrieved_date.iloc[0]} | {df.chembl_release.iloc[0]}')
    print(f'target    {df.target_chembl_id.iloc[0]} ({t["name"]})')
    if df.target_chembl_id.nunique() != 1 or df.target_chembl_id.iloc[0] != t['chembl_id']:
        raise ValueError(f'{source.name} does not hold {t["chembl_id"]} only')

column_sets = {tuple(s['df'].columns) for s in S.values()}
assert len(column_sets) == 1, 'the training CSVs do not share one column layout'
print(f'\ncolumns in every source file:\n  ' + '\n  '.join(next(iter(column_sets))))

## 1. Labels and potency

Cutoffs are from notebook 1: active ≥ 6.0, inactive < 5.0, the 5–6 band is `dropped`.
pActivity = 9 − log10(nM).

In [ ]:
for slug, s in S.items():
    banner(slug)
    df = s['df']
    counts = df.label.value_counts().reindex(LABELS, fill_value=0)
    usable = df[df.label.isin(['active', 'inactive'])].copy()

    label_table = counts.rename('molecules').to_frame()
    label_table['percent'] = (counts / len(df) * 100).round(2)
    display(label_table)

    always_active = counts['active'] / len(usable)
    print(f'binary training rows      {len(usable):,}')
    print(f'active:inactive ratio     {counts["active"] / counts["inactive"]:.2f} : 1')
    print(f'\nPredicting "active" for every row scores {always_active:.2%} accuracy')
    print(f'...but 50.00% balanced accuracy and 0% inactive recall.')

    print(f'\nmedian pActivity          {df.pActivity.median():.3f}'
          f'  (~{10 ** (9 - df.pActivity.median()):.1f} nM)')
    print(f'range                     {df.pActivity.min():.3f} - {df.pActivity.max():.3f}')
    s.update(counts=counts, usable=usable)

## 2. Completeness and consistency

In [ ]:
for slug, s in S.items():
    banner(slug)
    df = s['df']
    missing = df.isna().sum().rename('missing').to_frame()
    missing['percent'] = (missing.missing / len(df) * 100).round(2)
    display(missing[missing.missing > 0])
    print(f'columns with no missing values: {int((missing.missing == 0).sum())} of {len(missing)}')

    expected = np.where(df.pActivity >= 6, 'active',
                        np.where(df.pActivity < 5, 'inactive', 'dropped'))
    outside = ((df.pActivity < df.pActivity_min - 0.000501) |
               (df.pActivity > df.pActivity_max + 0.000501))
    exact_spread = df.pActivity_max - df.pActivity_min
    spread_mismatch = (df.pActivity_spread - exact_spread).abs() > 0.005001

    print(f'\nduplicate molecule IDs                      {int(df.molecule_chembl_id.duplicated().sum())}')
    print(f'duplicate input SMILES                      {int(df.canonical_smiles.duplicated().sum())}')
    print(f'labels that disagree with rounded pActivity  {int((df.label != expected).sum())}')
    print(f'medians outside min/max beyond tolerance    {int(outside.sum())}')
    print(f'spread inconsistent beyond tolerance        {int(spread_mismatch.sum())}')

    boundary = df.loc[df.label != expected].copy()
    boundary['label_from_rounded_pActivity'] = expected[df.label != expected]
    boundary['median_reconstructed_if_n_le_2'] = np.where(
        boundary.n_measurements <= 2,
        (boundary.pActivity_min + boundary.pActivity_max) / 2, np.nan)
    if len(boundary):
        display(boundary[['molecule_chembl_id', 'pActivity', 'label',
                          'label_from_rounded_pActivity', 'median_reconstructed_if_n_le_2']])
    s.update(expected=expected, outside=outside, exact_spread=exact_spread,
             spread_mismatch=spread_mismatch)

## 3. Measurement agreement

A molecule with a single measurement has a spread of zero by definition. 5-LOX has the most
repeated measurements (31% of molecules), then p38 (15%) and COX-2 (9%).

In [ ]:
for slug, s in S.items():
    banner(slug)
    df, exact_spread = s['df'], s['exact_spread']
    repeated = df.n_measurements > 1
    crosses_active = (df.pActivity_min < 6) & (df.pActivity_max >= 6)
    crosses_inactive = (df.pActivity_min < 5) & (df.pActivity_max >= 5)
    spans_classes = (df.pActivity_min < 5) & (df.pActivity_max >= 6)

    print(f'measurements represented in total   {int(df.n_measurements.sum()):,}')
    print(f'single-measurement molecules        {int((~repeated).sum()):,} ({(~repeated).mean():.1%})')
    print(f'molecules with repeats              {int(repeated.sum()):,}')
    print(f'\nspread >= 1 log unit (10-fold)      {int((exact_spread >= 1).sum()):,}')
    print(f'spread >= 2 log units (100-fold)    {int((exact_spread >= 2).sum()):,}')
    print(f'\nstraddle the active cutoff (6.0)    {int(crosses_active.sum()):,}')
    print(f'straddle the inactive cutoff (5.0)  {int(crosses_inactive.sum()):,}   (counts overlap)')
    print(f'span BOTH active and inactive       {int(spans_classes.sum()):,}   <-- the worrying one')
    s.update(repeated=repeated, crosses_active=crosses_active,
             crosses_inactive=crosses_inactive, spans_classes=spans_classes)

## 4. Descriptors

Seven descriptors computed from `canonical_smiles` with RDKit:

| Column | RDKit call |
|---|---|
| `molecular_weight` | `Descriptors.MolWt` |
| `cLogP` | `Crippen.MolLogP` |
| `TPSA` | `rdMolDescriptors.CalcTPSA` |
| `HBD` | `Lipinski.NumHDonors` |
| `HBA` | `Lipinski.NumHAcceptors` |
| `rotatable_bonds` | `Lipinski.NumRotatableBonds` |
| `ring_count` | `rdMolDescriptors.CalcNumRings` |

These are the Lipinski and Veber properties. The Murcko scaffold and fragment count come from
the same pass. Structures are parsed as supplied, before the standardization in notebook 3.

In [ ]:
DESCRIPTORS = ['molecular_weight', 'cLogP', 'TPSA', 'HBD', 'HBA', 'rotatable_bonds', 'ring_count']
VIOLATION_GROUPS = ['0', '1', '2', '3+']
descriptor_cols = DESCRIPTORS


def lipinski_violations(frame):
    return ((frame.molecular_weight > 500).astype(int) + (frame.cLogP > 5).astype(int)
            + (frame.HBD > 5).astype(int) + (frame.HBA > 10).astype(int))


def violation_group(frame):
    return lipinski_violations(frame).clip(upper=3).astype(int).astype(str).replace({'3': '3+'})


def describe_structures(smiles_series):
    rows = []
    for smiles in smiles_series:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            rows.append({'valid_smiles': False})
            continue
        rows.append({
            'valid_smiles': True,
            'rdkit_smiles': Chem.MolToSmiles(mol, isomericSmiles=True),
            'murcko_scaffold': MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False),
            'molecular_weight': Descriptors.MolWt(mol),
            'cLogP': Crippen.MolLogP(mol),
            'TPSA': rdMolDescriptors.CalcTPSA(mol),
            'HBD': Lipinski.NumHDonors(mol),
            'HBA': Lipinski.NumHAcceptors(mol),
            'rotatable_bonds': Lipinski.NumRotatableBonds(mol),
            'ring_count': rdMolDescriptors.CalcNumRings(mol),
            'fragment_count': len(Chem.GetMolFrags(mol)),
        })
    return pd.DataFrame(rows)


for slug, s in S.items():
    banner(slug)
    df = s['df']
    enriched = pd.concat([df, describe_structures(df.canonical_smiles)], axis=1)
    enriched['exact_activity_spread'] = s['exact_spread']
    enriched['crosses_active_cutoff'] = s['crosses_active']
    enriched['crosses_inactive_cutoff'] = s['crosses_inactive']
    enriched['spans_active_and_inactive'] = s['spans_classes']
    enriched['lipinski_violations'] = lipinski_violations(enriched)
    s['enriched'] = enriched

    print(f'computed {len(descriptor_cols)} descriptors for {len(enriched):,} molecules')
    print(f'invalid SMILES: {int((~enriched.valid_smiles).sum())}')
    display(enriched[['molecule_chembl_id', 'label', *descriptor_cols]].head())

### Descriptors by label

In [ ]:
for slug, s in S.items():
    banner(slug)
    enriched = s['enriched']
    print('median by label')
    display(enriched.groupby('label')[descriptor_cols].median().round(2).T)

    print('spread across the whole dataset')
    display(enriched[descriptor_cols].agg(['min', 'median', 'max']).round(2).T)

    lipinski = pd.DataFrame({
        'MW <= 500': enriched.molecular_weight.le(500),
        'cLogP <= 5': enriched.cLogP.le(5),
        'HBD <= 5': enriched.HBD.le(5),
        'HBA <= 10': enriched.HBA.le(10),
        'rot. bonds <= 10 (Veber)': enriched.rotatable_bonds.le(10),
        'TPSA <= 140 (Veber)': enriched.TPSA.le(140),
    })
    print('rule compliance')
    display((lipinski.mean() * 100).round(1).rename('% of molecules passing').to_frame())
    print(f'pass all four Lipinski criteria: '
          f'{lipinski.iloc[:, :4].all(axis=1).mean():.1%} of molecules')

Actives are slightly larger and greasier than inactives in all three datasets, but the
descriptors separate the labels only weakly. COX-2's polar descriptors (TPSA, HBD, HBA)
correlate slightly negatively with potency, unlike p38's. All correlations are small
(|ρ| ≤ 0.22).

A few very large molecules stretch the ranges: peptides in p38 and a series above 1,800 Da in
COX-2.

### Potency against descriptors

Spearman correlations, hexbin plots with the median per decile, and potency by the number of
Rule-of-Five criteria broken (3 means three or more).

In [ ]:
for slug, s in S.items():
    banner(slug)
    enriched = s['enriched']
    d = enriched.copy()
    d['group'] = pd.Categorical(violation_group(d), VIOLATION_GROUPS, ordered=True)
    rho = d[DESCRIPTORS].corrwith(d.pActivity, method='spearman').sort_values()

    descriptor_fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

    ax = axes[0, 0]
    ax.barh(rho.index, rho.values, color='#5d7b9d')
    ax.axvline(0, color='#444444', linewidth=.8)
    for y, value in enumerate(rho.values):
        ax.text(value + .005, y, f'{value:+.2f}', va='center', fontsize=9)
    ax.set(title='Rank correlation with pActivity', xlabel='Spearman rho',
           xlim=(min(0, rho.min() * 1.25 - .05), max(.3, rho.max() * 1.25)))

    for ax, column, limit, xlabel in [(axes[0, 1], 'molecular_weight', 500, 'Molecular weight (g/mol)'),
                                      (axes[0, 2], 'cLogP', 5, 'cLogP'),
                                      (axes[1, 0], 'TPSA', 140, 'TPSA (A^2)')]:
        lo, hi = d[column].quantile([.005, .995])
        inside = d[d[column].between(lo, hi)]
        ax.hexbin(inside[column], inside.pActivity, gridsize=34, bins='log', mincnt=1,
                  cmap='Blues', extent=(lo, hi, 3.8, 10.8))
        edges = inside[column].quantile(np.linspace(0, 1, 11)).unique()
        trend = (inside.groupby(pd.cut(inside[column], edges, include_lowest=True), observed=True)
                       .agg(x=(column, 'median'), y=('pActivity', 'median')))
        ax.plot(trend.x, trend.y, color='#b54c68', marker='o', markersize=4, linewidth=1.8,
                label='median per decile')
        ax.axvline(limit, color='#444444', linestyle='--', linewidth=1,
                   label=f'Lipinski/Veber limit ({limit})')
        for cutoff in [5, 6]:
            ax.axhline(cutoff, color='#444444', linestyle=':', linewidth=.8)
        ax.set(title=f'{xlabel.split(" (")[0]} vs potency '
                     f'(rho = {d[column].corr(d.pActivity, method="spearman"):+.2f})',
               xlabel=xlabel, ylabel='pActivity', xlim=(lo, hi))
        ax.legend(fontsize=8, loc='lower right')

    ax = axes[1, 1]
    data = [d.loc[d.group == g, 'pActivity'] for g in VIOLATION_GROUPS]
    box = ax.boxplot(data, tick_labels=[f'{g}\n(n={len(v):,})' for g, v in zip(VIOLATION_GROUPS, data)],
                     patch_artist=True, widths=.6, showfliers=False,
                     medianprops={'color': '#b54c68', 'linewidth': 1.6})
    for patch in box['boxes']:
        patch.set(facecolor='#cfd8e3', linewidth=.8)
    for cutoff in [5, 6]:
        ax.axhline(cutoff, color='#444444', linestyle=':', linewidth=.8)
    ax.set(title='Potency by Lipinski violations', xlabel='Rule-of-Five criteria broken',
           ylabel='pActivity')

    ax = axes[1, 2]
    mix = (pd.crosstab(d.group, d.label, normalize='index')
             .reindex(index=VIOLATION_GROUPS, columns=LABELS, fill_value=0) * 100)
    bottom = np.zeros(len(mix))
    for label, color in zip(LABELS, COLORS):
        ax.bar(mix.index, mix[label], bottom=bottom, color=color, label=label)
        for x, (b, v) in enumerate(zip(bottom, mix[label])):
            if v >= 3:
                ax.text(x, b + v / 2, f'{v:.0f}%', ha='center', va='center', fontsize=8, color='white')
        bottom += mix[label].values
    ax.set(title='Label mix by Lipinski violations', xlabel='Rule-of-Five criteria broken',
           ylabel='% of molecules', ylim=(0, 100))
    ax.legend(fontsize=8, loc='lower left')

    descriptor_fig.suptitle(f'NeuroLead - {slug} potency vs physicochemical descriptors', fontsize=19)
    plt.show()
    s['descriptor_fig'] = descriptor_fig
    s['rho'] = rho

    print('Lipinski criteria broken (MW>500, cLogP>5, HBD>5, HBA>10):')
    display(enriched.assign(violations=enriched.lipinski_violations.clip(upper=3))
            .groupby('violations').agg(molecules=('pActivity', 'size'),
                                       median_pActivity=('pActivity', 'median'),
                                       pct_active=('label', lambda x: (x == 'active').mean() * 100),
                                       pct_inactive=('label', lambda x: (x == 'inactive').mean() * 100)).round(2))

display(Markdown('#### Spearman rho with pActivity, all three targets'))
display(pd.DataFrame({slug: s['rho'] for slug, s in S.items()}).reindex(DESCRIPTORS).round(3))

In [ ]:
for slug, s in S.items():
    banner(slug)
    enriched = s['enriched']
    large = enriched[enriched.molecular_weight > 1000].sort_values('molecular_weight', ascending=False)
    s['large'] = large
    print(f'{len(large)} molecules exceed 1,000 Da, up to {enriched.molecular_weight.max():,.1f} Da')
    if len(large):
        display(large[['molecule_chembl_id', 'label', 'pActivity', *descriptor_cols]].round(2))
        display(Draw.MolsToGridImage(
            [Chem.MolFromSmiles(sm) for sm in large.canonical_smiles.head(3)],
            legends=[f'{r.molecule_chembl_id} | {r.molecular_weight:,.0f} Da'
                     for r in large.head(3).itertuples()],
            molsPerRow=3, subImgSize=(300, 260)))

## 5. Endpoint type

In [ ]:
audit_columns = ['molecule_chembl_id', 'molecule_pref_name', 'label', 'standard_type',
                 'pActivity', 'pActivity_min', 'pActivity_max', 'n_measurements',
                 'exact_activity_spread', 'crosses_active_cutoff',
                 'crosses_inactive_cutoff', 'spans_active_and_inactive']

for slug, s in S.items():
    banner(slug)
    df, enriched, exact_spread = s['df'], s['enriched'], s['exact_spread']
    assay_table = pd.crosstab(df.standard_type, df.label).reindex(columns=LABELS, fill_value=0)
    display(assay_table)

    flagged = (enriched[(exact_spread >= 1) | s['crosses_active'] | s['crosses_inactive']]
               .sort_values('exact_activity_spread', ascending=False))
    s['flagged'] = flagged
    print(f'{len(flagged):,} molecules flagged for measurement review')
    display(flagged[audit_columns[:8] + ['exact_activity_spread']].head(10).round(3))

## 6. Chemical diversity

Murcko scaffolds, with stereochemistry ignored.

In [ ]:
for slug, s in S.items():
    banner(slug)
    df, enriched = s['df'], s['enriched']
    sc = enriched.loc[enriched.valid_smiles & enriched.murcko_scaffold.ne('')].copy()
    scaffolds = sc.groupby('murcko_scaffold').agg(
        molecules=('molecule_chembl_id', 'size'),
        active=('label', lambda x: int((x == 'active').sum())),
        inactive=('label', lambda x: int((x == 'inactive').sum())),
        dropped=('label', lambda x: int((x == 'dropped').sum())),
    ).sort_values('molecules', ascending=False)
    s.update(sc=sc, scaffolds=scaffolds)

    print(f'distinct nonempty scaffolds, all rows      {len(scaffolds):,}')
    print(f'distinct nonempty scaffolds, binary rows   '
          f'{int(sc.loc[sc.label != "dropped", "murcko_scaffold"].nunique()):,}')
    print(f'scaffolds containing an inactive           {int((scaffolds.inactive > 0).sum()):,}')
    print(f'acyclic molecules (empty scaffold)         {int(enriched.murcko_scaffold.eq("").sum())}')
    print(f'molecules in the top 10 scaffolds          {int(scaffolds.head(10).molecules.sum()):,}'
          f' ({scaffolds.head(10).molecules.sum() / len(df):.1%} of all rows)')
    display(scaffolds.head(10))

    display(Draw.MolsToGridImage(
        [Chem.MolFromSmiles(sm) for sm in scaffolds.head(6).index],
        legends=[f'{n:,} molecules' for n in scaffolds.head(6).molecules],
        molsPerRow=3, subImgSize=(280, 220)))

### Duplicate structures

Checked on canonical SMILES as supplied, so salts are not stripped here.

In [ ]:
for slug, s in S.items():
    banner(slug)
    enriched = s['enriched']
    duplicate_structures = enriched[enriched.rdkit_smiles.notna()
                                    & enriched.rdkit_smiles.duplicated(keep=False)]
    s['duplicate_structures'] = duplicate_structures
    print(f'rows sharing a canonical structure   {len(duplicate_structures)}')
    print(f'molecules with multiple fragments    {int((enriched.fragment_count > 1).sum())}'
          f'   <- salts and mixtures, not deduplicated here')
    if len(duplicate_structures):
        display(duplicate_structures[['molecule_chembl_id', 'label', 'pActivity', 'rdkit_smiles']])

### Browse molecules

Active molecules with at least two measurements, most potent first.

In [ ]:
selected_label = 'active'
minimum_pactivity = 4.0
maximum_pactivity = 11.0
minimum_measurements = 2

for slug, s in S.items():
    banner(slug)
    enriched = s['enriched']
    mask = (enriched.pActivity.between(minimum_pactivity, maximum_pactivity)
            & enriched.n_measurements.ge(minimum_measurements))
    if selected_label is not None:
        mask &= enriched.label.eq(selected_label)
    selected = enriched.loc[mask].sort_values('pActivity', ascending=False)

    print(f'{len(selected):,} molecules match')
    display(selected[['molecule_chembl_id', 'molecule_pref_name', 'label', 'standard_type',
                      'pActivity', 'n_measurements', 'exact_activity_spread',
                      *descriptor_cols]].head(20).round(2))

    preview = selected.head(8)
    if len(preview):
        display(Draw.MolsToGridImage(
            [Chem.MolFromSmiles(sm) for sm in preview.canonical_smiles],
            legends=[f'{r.molecule_chembl_id} | pAct {r.pActivity:.2f}' for r in preview.itertuples()],
            molsPerRow=4, subImgSize=(260, 220)))

## 7. Overview figure

In [ ]:
for slug, s in S.items():
    banner(slug)
    df, enriched, counts = s['df'], s['enriched'], s['counts']
    repeated, exact_spread, scaffolds = s['repeated'], s['exact_spread'], s['scaffolds']

    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

    ax = axes[0, 0]
    bars = ax.bar(LABELS, counts, color=COLORS)
    ax.bar_label(bars, labels=[f'{n:,}\n({n / len(df):.1%})' for n in counts], padding=3)
    ax.set(ylim=(0, counts.max() * 1.22), title='Label counts: full dataset', ylabel='Molecules')

    ax = axes[0, 1]
    bins = np.linspace(3.8, 10.8, 36)
    for label, color in zip(LABELS, COLORS):
        ax.hist(df.loc[df.label == label, 'pActivity'], bins=bins, color=color, label=label, alpha=.8)
    for cutoff in [5, 6]:
        ax.axvline(cutoff, color='#444444', linestyle='--', linewidth=1)
    ax.set(title='Potency and fixed label cutoffs',
           xlabel='pActivity (higher = more potent)', ylabel='Molecules')

    ax = axes[0, 2]
    repeat_bins = pd.cut(df.n_measurements, [0, 1, 2, 5, 10, np.inf],
                         labels=['1', '2', '3-5', '6-10', '11+'])
    bars = ax.bar(repeat_bins.cat.categories, repeat_bins.value_counts(sort=False), color='#5d7b9d')
    ax.bar_label(bars, padding=3)
    ax.set(title='Measurement coverage', xlabel='Measurements per molecule', ylabel='Molecules',
           ylim=(0, (~repeated).sum() * 1.15))

    ax = axes[1, 0]
    ax.hist(exact_spread[repeated], bins=np.arange(0, 4.01, .2), color='#5d7b9d')
    ax.axvline(1, color='#b54c68', linestyle='--', label='10-fold range')
    ax.set(title=f'Disagreement: {repeated.sum():,} repeated molecules',
           xlabel='Max - min pActivity (log units)', ylabel='Molecules')
    ax.legend()

    ax = axes[1, 1]
    for label, color in zip(LABELS, COLORS):
        part = enriched[enriched.label == label]
        ax.scatter(part.molecular_weight, part.TPSA, s=9, alpha=.35, c=color,
                   label=label, rasterized=True)
    ax.set(title='Chemical property coverage', xlabel='Molecular weight (g/mol)', ylabel='TPSA (A^2)')
    ax.legend(markerscale=2)

    ax = axes[1, 2]
    top = scaffolds.head(10)
    ax.barh([f'Scaffold {i}' for i in range(1, len(top) + 1)], top.molecules, color='#5d7b9d')
    ax.invert_yaxis()
    ax.set(title='Most frequent Murcko scaffolds', xlabel='Molecules (including dropped)')

    fig.suptitle(f'NeuroLead - {slug} training data exploration', fontsize=19)
    plt.show()
    s['fig'] = fig

## 8. Metrics

Headline numbers for each target, saved to `analysis/{slug}/summary.json` together with the
SHA-256 of the source file.

In [ ]:
for slug, s in S.items():
    df, enriched, counts, usable = s['df'], s['enriched'], s['counts'], s['usable']
    repeated, exact_spread = s['repeated'], s['exact_spread']
    sc, scaffolds = s['sc'], s['scaffolds']
    s['metrics'] = {
        'target_chembl_id': TARGETS[slug]['chembl_id'],
        'target_name': TARGETS[slug]['name'],
        'rows': len(df), 'columns': len(df.columns),
        'labels': {k: int(v) for k, v in counts.items()},
        'binary_training_rows': len(usable),
        'active_inactive_ratio': float(counts['active'] / counts['inactive']),
        'always_active_accuracy': float(counts['active'] / len(usable)),
        'always_active_balanced_accuracy': 0.5,
        'pActivity_median': float(df.pActivity.median()),
        'median_implied_potency_nM': float(10 ** (9 - df.pActivity.median())),
        'measurements_total': int(df.n_measurements.sum()),
        'single_measurement_molecules': int((~repeated).sum()),
        'repeated_measurement_molecules': int(repeated.sum()),
        'spread_at_least_1_log_unit': int((exact_spread >= 1).sum()),
        'spread_at_least_2_log_units': int((exact_spread >= 2).sum()),
        'crosses_active_cutoff': int(s['crosses_active'].sum()),
        'crosses_inactive_cutoff': int(s['crosses_inactive'].sum()),
        'spans_active_and_inactive': int(s['spans_classes'].sum()),
        'invalid_smiles': int((~enriched.valid_smiles).sum()),
        'duplicate_molecule_ids': int(df.molecule_chembl_id.duplicated().sum()),
        'duplicate_original_smiles': int(df.canonical_smiles.duplicated().sum()),
        'duplicate_rdkit_structure_rows': len(s['duplicate_structures']),
        'rounded_label_mismatches': int((df.label != s['expected']).sum()),
        'median_outside_range_beyond_rounding': int(s['outside'].sum()),
        'spread_rounding_mismatches': int(s['spread_mismatch'].sum()),
        'unique_nonempty_scaffolds_all_rows': len(scaffolds),
        'unique_nonempty_scaffolds_binary_rows': int(
            sc.loc[sc.label != 'dropped', 'murcko_scaffold'].nunique()),
        'scaffolds_containing_inactives': int((scaffolds.inactive > 0).sum()),
        'acyclic_molecules_empty_scaffold': int(enriched.murcko_scaffold.eq('').sum()),
        'top_10_scaffold_molecules': int(scaffolds.head(10).molecules.sum()),
        'multi_fragment_molecules': int((enriched.fragment_count > 1).sum()),
        'molecules_above_1000_Da': len(s['large']),
        'confidence_counts': {str(k): int(v) for k, v in df.confidence_score.value_counts().items()},
        'source_sha256': hashlib.sha256(s['source'].read_bytes()).hexdigest(),
        'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                     'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
                     'rdkit': rdBase.rdkitVersion},
    }

side_by_side = pd.DataFrame({
    slug: {k: v for k, v in s['metrics'].items()
           if not isinstance(v, dict) and k not in ('source_sha256', 'target_name')}
    for slug, s in S.items()})
for lab in LABELS:
    side_by_side.loc[f'labels: {lab}'] = [s['metrics']['labels'][lab] for s in S.values()]
pd.set_option('display.max_rows', 60)
display(side_by_side)

## 9. Output files

In [ ]:
if not WRITE_OUTPUTS:
    print('WRITE_OUTPUTS is False, so nothing was written.')
else:
    for slug, s in S.items():
        out = s['out']
        out.mkdir(parents=True, exist_ok=True)
        written = []
        for frame, name, index in [
            (s['enriched'], 'molecules_with_descriptors.csv', False),
            (s['flagged'][audit_columns], 'measurement_review.csv', False),
            (s['scaffolds'], 'scaffold_summary.csv', True),
        ]:
            frame.to_csv(out / name, index=index)
            written.append(name)

        (out / 'summary.json').write_text(json.dumps(s['metrics'], indent=2), encoding='utf-8')
        written.append('summary.json')
        s['fig'].savefig(out / 'overview.png', dpi=180)
        written.append('overview.png')
        s['descriptor_fig'].savefig(out / 'descriptor_vs_activity.png', dpi=180)
        written.append('descriptor_vs_activity.png')

        print(f'wrote {len(written)} files to {out.relative_to(ROOT).as_posix()}/: '
              + ', '.join(written))